In [1]:
import sys
!{sys.executable} -m pip install onnx onnxruntime psutil py-cpuinfo onnxscript

  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/7.9 MB 2.8 MB/s eta 0:00:03
   -------------- ------------------------- 2.9/7.9 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 7.9/7.9 MB 15.2 MB/s  0:00:00
   ---------------------------------------- 0.0/14.3 MB ? eta -:--:--
   --------- ------------------------------ 3.4/14.3 MB 18.3 MB/s eta 0:00:01
   ------------- -------------------------- 5.0/14.3 MB 12.1 MB/s eta 0:00:01
   ------------------ --------------------- 6.6/14.3 MB 10.6 MB/s eta 0:00:01
   ---------------------- ----------------- 8.1/14.3 MB 10.1 MB/s eta 0:00:01
   ---------------------------- ----------- 10.2/14.3 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------  14.2/14.3 MB 11.1 MB/s eta 0:00:01
   -------------------

In [1]:
import sys, time, json, csv, random, statistics, platform, datetime
from pathlib import Path
import numpy as np
import torch
import onnxruntime as ort
import psutil
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

MODEL_NAME = "mobilenet_v2"          # later you can try "resnet18"
DATA = Path("../data/imagenette2-160")   # same data as Phases 3 and 4
OUT = Path("../models/phase6")
RESULTS = Path("../../results")
OUT.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

FP32_PATH = OUT / f"{MODEL_NAME}_fp32.onnx"
INT8_PATH = OUT / f"{MODEL_NAME}_int8.onnx"
print("Setup done.")

Setup done.


In [2]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load_model(name):
    if name == "mobilenet_v2":
        m = models.mobilenet_v2(weights="DEFAULT")
    elif name == "resnet18":
        m = models.resnet18(weights="DEFAULT")
    else:
        raise ValueError(name)
    return m.eval()

In [4]:
model = load_model(MODEL_NAME)
dummy = torch.randn(1, 3, 224, 224)
kwargs = dict(input_names=["input"], output_names=["output"], opset_version=17)
try:
    torch.onnx.export(model, dummy, str(FP32_PATH), dynamo=False, **kwargs)
except Exception as e:
    print("Legacy exporter failed, trying default:", type(e).__name__)
    torch.onnx.export(model, dummy, str(FP32_PATH), **kwargs)
print("Saved", FP32_PATH)

C:\Users\user\AppData\Local\Temp\ipykernel_35432\3294312963.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, str(FP32_PATH), dynamo=False, **kwargs)


Saved ..\models\phase6\mobilenet_v2_fp32.onnx


In [5]:
from onnxruntime.quantization import (
    CalibrationDataReader, QuantFormat, QuantType, quantize_static)

class CalibrationReader(CalibrationDataReader):
    """Gives 100 real images so quantization learns each layer's value range."""
    def __init__(self, n):
        ds = ImageFolder(str(DATA / "train"), transform=transform)
        g = torch.Generator().manual_seed(0)
        idx = torch.randperm(len(ds), generator=g)[:n].tolist()
        self._it = iter([{"input": ds[i][0].unsqueeze(0).numpy().astype(np.float32)} for i in idx])
    def get_next(self):
        return next(self._it, None)

PREP_PATH = OUT / f"{MODEL_NAME}_fp32_prep.onnx"
src = FP32_PATH
try:
    from onnxruntime.quantization.shape_inference import quant_pre_process
    quant_pre_process(str(FP32_PATH), str(PREP_PATH))
    src = PREP_PATH
except Exception as e:
    print("Pre-processing skipped:", type(e).__name__)

quantize_static(
    str(src), str(INT8_PATH), CalibrationReader(100),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
)
for p in (FP32_PATH, INT8_PATH):
    print(p.name, f"{p.stat().st_size / (1024*1024):.2f} MB")

mobilenet_v2_fp32.onnx 13.34 MB
mobilenet_v2_int8.onnx 3.72 MB


In [3]:
WNID_TO_IMAGENET = {
    "n01440764": 0, "n02102040": 217, "n02979186": 482, "n03000684": 491,
    "n03028079": 497, "n03394916": 566, "n03417042": 569, "n03425413": 571,
    "n03445777": 574, "n03888257": 701,
}
val_ds = ImageFolder(str(DATA / "val"), transform=transform)
idx = [WNID_TO_IMAGENET[w] for w in val_ds.classes]
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

def onnx_accuracy(path):
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    correct = total = 0
    for images, labels in val_loader:
        logits = sess.run(None, {name: images.numpy().astype(np.float32)})[0]
        pred = np.argmax(logits[:, idx], axis=1)
        correct += int((pred == labels.numpy()).sum())
        total += len(labels)
    return 100 * correct / total

for kind, p in (("fp32", FP32_PATH), ("int8", INT8_PATH)):
    print(kind, f"{onnx_accuracy(p):.2f}%")

fp32 97.35%
int8 97.27%


In [3]:
TRIALS = 10          # how many repeated rounds
RUNS = 50            # timed runs inside each round
WARMUP = 10          # throwaway runs before timing
THREADS = [1, 4, 8]  # how many CPU threads to test
AFFINITY = None      # later: "0-11" to use only the fast cores
TAG = "laptopA_repeat"
USE_OPENVINO = False

def parse_cpus(s):
    cpus = []
    for part in s.split(","):
        if "-" in part:
            a, b = part.split("-")
            cpus += list(range(int(a), int(b) + 1))
        else:
            cpus.append(int(part))
    return cpus

if AFFINITY:
    cpus = parse_cpus(AFFINITY)
    psutil.Process().cpu_affinity(cpus)
    THREADS = [t for t in THREADS if t <= len(cpus)]
    print("Pinned to logical CPUs:", cpus)

try:
    import cpuinfo
    cpu_name = cpuinfo.get_cpu_info().get("brand_raw", platform.processor())
except Exception:
    cpu_name = platform.processor()

batt = psutil.sensors_battery()
info = {
    "tag": TAG, "model": MODEL_NAME, "cpu": cpu_name,
    "logical_cpus": psutil.cpu_count(logical=True),
    "os": platform.platform(), "python": platform.python_version(),
    "torch": torch.__version__, "onnxruntime": ort.__version__,
    "plugged_in": None if batt is None else batt.power_plugged,
    "affinity": AFFINITY, "trials": TRIALS, "runs_per_trial": RUNS,
}
print(json.dumps(info, indent=2))
if info["plugged_in"] is False:
    print("WARNING: on battery. Plug in and use best-performance power mode.")

x_np = np.random.default_rng(0).standard_normal((1, 3, 224, 224)).astype(np.float32)

{
  "tag": "laptopA_repeat",
  "model": "mobilenet_v2",
  "cpu": "13th Gen Intel(R) Core(TM) i7-13620H",
  "logical_cpus": 16,
  "os": "Windows-10-10.0.26200-SP0",
  "python": "3.11.15",
  "torch": "2.11.0+cu126",
  "onnxruntime": "1.30.0",
  "plugged_in": true,
  "affinity": null,
  "trials": 10,
  "runs_per_trial": 50
}


In [4]:
class Config:
    def __init__(self, runtime, precision, threads, prepare, run):
        self.runtime, self.precision, self.threads = runtime, precision, threads
        self.prepare, self.run = prepare, run
        self.name = f"{runtime}_{precision}_t{threads}"

def make_torch(threads):
    m = load_model(MODEL_NAME)
    x = torch.from_numpy(x_np)
    def prepare():
        torch.set_num_threads(threads)
    def run():
        with torch.inference_mode():
            m(x)
    return Config("pytorch", "fp32", threads, prepare, run)

def make_ort(path, precision, threads):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
    feed = {sess.get_inputs()[0].name: x_np}
    return Config("onnxruntime", precision, threads,
                  lambda: None, lambda: sess.run(None, feed))

def make_ov(path, threads):
    import openvino as ov
    core = ov.Core()
    compiled = core.compile_model(
        core.read_model(str(path)), "CPU",
        {"INFERENCE_NUM_THREADS": threads, "PERFORMANCE_HINT": "LATENCY"})
    return Config("openvino", "fp32", threads, lambda: None, lambda: compiled(x_np))

configs = []
for t in THREADS:
    configs.append(make_torch(t))
    configs.append(make_ort(FP32_PATH, "fp32", t))
    configs.append(make_ort(INT8_PATH, "int8", t))
    if USE_OPENVINO:
        configs.append(make_ov(FP32_PATH, t))
print(len(configs), "configurations ready:", [c.name for c in configs])

9 configurations ready: ['pytorch_fp32_t1', 'onnxruntime_fp32_t1', 'onnxruntime_int8_t1', 'pytorch_fp32_t4', 'onnxruntime_fp32_t4', 'onnxruntime_int8_t4', 'pytorch_fp32_t8', 'onnxruntime_fp32_t8', 'onnxruntime_int8_t8']


In [5]:
def time_block(fn, runs):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000.0)   # milliseconds
    return statistics.median(times)

def bootstrap_ci(vals, n=2000, seed=0):
    r = random.Random(seed)
    meds = sorted(statistics.median(r.choices(vals, k=len(vals))) for _ in range(n))
    return meds[int(0.025 * n)], meds[int(0.975 * n) - 1]

In [6]:
print("Initial warm-up ...")
for c in configs:
    c.prepare()
    for _ in range(30):
        c.run()

trial_meds = {c.name: [] for c in configs}
rng = random.Random(0)
for trial in range(TRIALS):
    order = configs[:]
    rng.shuffle(order)            # mix the order so heat affects everyone equally
    for c in order:
        c.prepare()
        for _ in range(WARMUP):
            c.run()
        trial_meds[c.name].append(time_block(c.run, RUNS))
    print(f"trial {trial + 1}/{TRIALS} done")

Initial warm-up ...
trial 1/10 done
trial 2/10 done
trial 3/10 done
trial 4/10 done
trial 5/10 done
trial 6/10 done
trial 7/10 done
trial 8/10 done
trial 9/10 done
trial 10/10 done


In [7]:
rows = []
for c in configs:
    v = trial_meds[c.name]
    med = statistics.median(v)
    lo, hi = bootstrap_ci(v)
    rows.append({
        "name": c.name, "runtime": c.runtime, "precision": c.precision,
        "threads": c.threads, "median_ms": med, "ci_low_ms": lo, "ci_high_ms": hi,
        "cv_percent": 100 * statistics.pstdev(v) / statistics.mean(v),
    })

ref = {r["threads"]: r for r in rows if r["runtime"] == "pytorch"}
for r in rows:
    base = ref[r["threads"]]
    overlap = not (r["ci_high_ms"] < base["ci_low_ms"] or r["ci_low_ms"] > base["ci_high_ms"])
    r["vs_pytorch_x"] = base["median_ms"] / r["median_ms"]
    if r is base:
        r["verdict"] = "reference"
    elif overlap:
        r["verdict"] = "within noise"
    else:
        r["verdict"] = "faster" if r["median_ms"] < base["median_ms"] else "slower"

print(f"{'config':28s} {'median ms':>10s} {'95% CI':>18s} {'CV%':>6s} {'x vs torch':>11s}  verdict")
for r in rows:
    ci = f"[{r['ci_low_ms']:.2f}, {r['ci_high_ms']:.2f}]"
    print(f"{r['name']:28s} {r['median_ms']:10.2f} {ci:>18s} {r['cv_percent']:6.1f} "
          f"{r['vs_pytorch_x']:11.2f}  {r['verdict']}")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
stem = RESULTS / f"phase6_{MODEL_NAME}_{TAG}_{stamp}"
with open(f"{stem}.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
with open(f"{stem}.json", "w") as f:
    json.dump(info, f, indent=2)
print("Saved", stem)

config                        median ms             95% CI    CV%  x vs torch  verdict
pytorch_fp32_t1                   56.10     [54.40, 59.27]    4.2        1.00  reference
onnxruntime_fp32_t1               28.73     [27.93, 29.95]    4.0        1.95  faster
onnxruntime_int8_t1               21.07     [20.47, 22.05]    4.9        2.66  faster
pytorch_fp32_t4                   46.47     [46.11, 47.41]   14.5        1.00  reference
onnxruntime_fp32_t4                9.48       [9.35, 9.86]    4.0        4.90  faster
onnxruntime_int8_t4                6.84       [6.64, 8.79]   17.0        6.80  faster
pytorch_fp32_t8                   51.66     [50.79, 52.14]    1.4        1.00  reference
onnxruntime_fp32_t8               50.65     [47.29, 52.94]    5.2        1.02  within noise
onnxruntime_int8_t8               34.68     [34.30, 37.32]    7.7        1.49  faster
Saved ..\..\results\phase6_mobilenet_v2_laptopA_repeat_20261007_1100
